# 05 - Model Evaluation and Persistence

**Objectif :** évaluer le modèle sélectionné, produire calibration/fairness, puis sauvegarder explicitement le bundle.

In [ ]:
import pandas as pd
import plotly.express as px

from credit_risk_lab.config.settings import settings

print(f"Project root: {settings.project_root}")
print(f"Environment: {settings.environment}")

## 1. Load training partition

In [ ]:
from credit_risk_lab.infrastructure.data_sources import CsvLoanDataLoader

train_raw = CsvLoanDataLoader(path=settings.train_path).load()
train_raw.shape

## 2. Rebuild features and development split

In [ ]:
from credit_risk_lab.application import three_way_stratified_split
from credit_risk_lab.infrastructure.feature_engineering import LoanFeatureEngineer

feature_engineer = LoanFeatureEngineer()
train_features = feature_engineer.transform(train_raw)
split = three_way_stratified_split(train_features)

sensitive_columns = [
    column for column in settings.sensitive_columns if column in split.x_train
]
x_train = split.x_train.drop(columns=sensitive_columns)
x_validation = split.x_validation.drop(columns=sensitive_columns)
x_test = split.x_test.drop(columns=sensitive_columns)
sensitive_test = split.x_test[sensitive_columns].copy()

## 3. Preprocess and train candidates

In [ ]:
from credit_risk_lab.infrastructure.modeling import (
    BestModelSelector,
    BoostingModelTrainer,
    CreditRiskPreprocessor,
)

preprocessor = CreditRiskPreprocessor()
x_train_t = preprocessor.fit_transform(x_train)
x_validation_t = preprocessor.transform(x_validation)
x_test_t = preprocessor.transform(x_test)

trainer = BoostingModelTrainer(random_state=settings.random_state)
training_results = trainer.fit(
    x_train_t,
    split.y_train,
    x_validation_t,
    split.y_validation,
)
best_result = BestModelSelector(metric=settings.selection_metric).select(training_results)
best_result.model_name

## 4. Evaluate on isolated internal test

In [ ]:
from credit_risk_lab.infrastructure.evaluation import CreditRiskModelEvaluator

evaluator = CreditRiskModelEvaluator()
test_probabilities = best_result.model.predict_proba(x_test_t)
test_metrics = evaluator.metrics_frame(
    best_result.model_name,
    split.y_test,
    test_probabilities,
    best_result.threshold,
)
test_metrics.round(4)

## 5. Calibration

In [ ]:
from credit_risk_lab.infrastructure.evaluation import CalibrationEvaluator
from credit_risk_lab.infrastructure.visualization import plot_calibration

calibration = CalibrationEvaluator(bins=10).evaluate(
    split.y_test,
    test_probabilities,
)

display(calibration.round(4))
plot_calibration(calibration, best_result.model_name).show()

## 6. Fairness diagnostics

In [ ]:
from credit_risk_lab.infrastructure.evaluation import FairnessEvaluator

fairness = FairnessEvaluator(min_group_size=30).evaluate(
    split.y_test,
    test_probabilities,
    sensitive_test,
    best_result.threshold,
)
fairness

## 7. Build metadata

In [ ]:
from credit_risk_lab.application.workflows import current_git_commit
from credit_risk_lab.infrastructure.modeling import sha256_file

metadata = {
    "model_name": best_result.model_name,
    "test_metrics": test_metrics.iloc[0].to_dict(),
    "selection_metric": settings.selection_metric,
    "split_strategy": settings.split_strategy,
    "training_dataset_sha256": sha256_file(settings.train_path),
    "external_test_dataset_sha256": sha256_file(settings.raw_test_path),
    "models_config_sha256": sha256_file(settings.models_config_path),
    "git_commit": current_git_commit(),
    "target_definition": "loan_status=1 is the synthetic positive risk class",
}

metadata

## 8. Save model bundle explicitly

In [ ]:
from credit_risk_lab.infrastructure.modeling import JoblibModelBundleRepository

repository = JoblibModelBundleRepository()
bundle_path = repository.save(
    settings.model_bundle_path,
    model=best_result.model,
    preprocessor=preprocessor.transformer,
    threshold=best_result.threshold,
    metadata=metadata,
)

bundle_path